# 🧹 Sprint 1 — Data Preprocessing

**Objective:** Clean the raw data, handle nulls, remove leakage columns, encode categoricals, scale numerics, and produce a model-ready dataset.

---

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load_data import load_raw_data, save_data
from src.data.clean_data import clean_data
from src.features.encode import label_encode, one_hot_encode, frequency_encode
from src.features.scale import standard_scale
from src.utils.config import (
    PROCESSED_DATA_DIR, NUMERIC_FEATURES,
    CATEGORICAL_LOW_CARD, CATEGORICAL_HIGH_CARD, TARGET
)

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Raw Data

In [2]:
df_raw = load_raw_data()
print(f'Raw shape: {df_raw.shape}')
df_raw.head()

2026-08-25 00:49:52 │ INFO     │ src.data.load_data │ Loading raw data from C:\Users\pares\OneDrive\Desktop\hotle_cnacelation\hotel_bookings.csv
2026-08-25 00:49:53 │ INFO     │ src.data.load_data │ Shape: (119390, 32)
2026-08-25 00:49:53 │ INFO     │ src.data.load_data │ Columns: ['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date']
2026-08-25 00:49:53 │ WARNING  │ src.data.load_data │ Columns with nulls:
children         4
countr

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


## 2. Clean Data
- Drop leakage columns (`reservation_status`, `reservation_status_date`)
- Drop high-null column (`company`)
- Impute `children`, `country`, `agent`
- Remove invalid rows

In [3]:
df_clean = clean_data(df_raw)
print(f'Cleaned shape: {df_clean.shape}')
print(f'\nRemaining nulls:\n{df_clean.isnull().sum()[df_clean.isnull().sum() > 0]}')
df_clean.head()

2026-08-25 00:49:55 │ INFO     │ src.data.clean_data │ Starting data cleaning  (input shape: (119390, 32))
2026-08-25 00:49:55 │ INFO     │ src.data.clean_data │ Dropped columns: ['reservation_status', 'reservation_status_date', 'company']
2026-08-25 00:49:55 │ INFO     │ src.data.clean_data │ Null counts after imputation:
Series([], )


c:\Users\pares\OneDrive\Desktop\hotle_cnacelation\src\data\clean_data.py:68: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["children"].fillna(0, inplace=True)
c:\Users\pares\OneDrive\Desktop\hotle_cnacelation\src\data\clean_data.py:71: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy

2026-08-25 00:49:55 │ INFO     │ src.data.clean_data │ Removed 181 invalid rows (negative ADR / zero guests)
2026-08-25 00:49:55 │ INFO     │ src.data.clean_data │ Converted arrival_date_month to numeric (1–12)
2026-08-25 00:49:56 │ INFO     │ src.data.clean_data │ Dropped 32246 duplicate rows
2026-08-25 00:49:57 │ INFO     │ src.data.clean_data │ Cleaned data saved to C:\Users\pares\OneDrive\Desktop\hotle_cnacelation\data\interim\hotel_bookings_cleaned.csv  (output shape: (86963, 29))
Cleaned shape: (86963, 29)

Remaining nulls:
Series([], dtype: int64)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
0,Resort Hotel,0,342,2015,7,27,1,0,0,2,...,C,C,3,No Deposit,0,0,Transient,0.0,0,0
1,Resort Hotel,0,737,2015,7,27,1,0,0,2,...,C,C,4,No Deposit,0,0,Transient,0.0,0,0
2,Resort Hotel,0,7,2015,7,27,1,0,1,1,...,A,C,0,No Deposit,0,0,Transient,75.0,0,0
3,Resort Hotel,0,13,2015,7,27,1,0,1,1,...,A,A,0,No Deposit,304,0,Transient,75.0,0,0
4,Resort Hotel,0,14,2015,7,27,1,0,2,2,...,A,A,0,No Deposit,240,0,Transient,98.0,0,1


## 3. Verify Cleaning

In [4]:
# Confirm leakage columns removed
assert 'reservation_status' not in df_clean.columns
assert 'reservation_status_date' not in df_clean.columns
assert 'company' not in df_clean.columns
print('✅ Leakage/high-null columns removed')

# Confirm no negative ADR
assert df_clean['adr'].min() >= 0
print('✅ No negative ADR')

# Confirm no zero-guest rows
guest_sum = df_clean[['adults', 'children', 'babies']].sum(axis=1)
assert guest_sum.min() > 0
print('✅ No zero-guest bookings')

# Confirm arrival_date_month is numeric
assert df_clean['arrival_date_month'].dtype in ['int64', 'int32', 'float64']
print('✅ arrival_date_month converted to numeric')

print(f'\nFinal shape: {df_clean.shape}')

✅ Leakage/high-null columns removed
✅ No negative ADR
✅ No zero-guest bookings
✅ arrival_date_month converted to numeric

Final shape: (86963, 29)


## 4. Encoding Demonstration

In [5]:
# Frequency-encode high-cardinality columns
df_encoded, freq_maps = frequency_encode(df_clean.copy(), CATEGORICAL_HIGH_CARD)
print(f'After frequency encoding: {df_encoded.shape}')
print(f'Country top-5 frequencies:\n{freq_maps["country"].head()}')

2026-08-25 00:49:57 │ INFO     │ src.features.encode │ Frequency-encoded 'country'  (178 categories)
After frequency encoding: (86963, 29)
Country top-5 frequencies:
country
PRT    0.312650
GBR    0.119626
FRA    0.101376
ESP    0.083185
DEU    0.061865
Name: proportion, dtype: float64


In [6]:
# One-hot encode low-cardinality categoricals
df_encoded = one_hot_encode(df_encoded, CATEGORICAL_LOW_CARD)
print(f'After one-hot encoding: {df_encoded.shape}')
df_encoded.head()

2026-08-25 00:49:57 │ INFO     │ src.features.encode │ One-hot encoded 8 columns → 60 total columns
After one-hot encoding: (86963, 60)


,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,...,assigned_room_type_G,assigned_room_type_H,assigned_room_type_I,assigned_room_type_K,assigned_room_type_L,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Group,customer_type_Transient,customer_type_Transient-Party
0,0,342,2015,7,27,1,0,0,2,0,...,0,0,0,0,0,0,0,0,1,0
1,0,737,2015,7,27,1,0,0,2,0,...,0,0,0,0,0,0,0,0,1,0
2,0,7,2015,7,27,1,0,1,1,0,...,0,0,0,0,0,0,0,0,1,0
3,0,13,2015,7,27,1,0,1,1,0,...,0,0,0,0,0,0,0,0,1,0
4,0,14,2015,7,27,1,0,2,2,0,...,0,0,0,0,0,0,0,0,1,0


## 5. Scaling Demonstration

In [7]:
# Identify numeric cols still present
num_cols = [c for c in NUMERIC_FEATURES if c in df_encoded.columns]

df_scaled, scaler = standard_scale(df_encoded, num_cols)
print(f'Scaled shape: {df_scaled.shape}')
df_scaled[num_cols].describe().T[['mean', 'std']]

2026-08-25 00:49:58 │ INFO     │ src.features.scale │ StandardScaler applied to 18 columns
Scaled shape: (86963, 60)


,mean,std
lead_time,-4.967745e-17,1.000006
arrival_date_year,4.436457e-14,1.000006
arrival_date_week_number,8.758918e-17,1.000006
arrival_date_day_of_month,7.647712e-17,1.000006
stays_in_weekend_nights,4.183364e-17,1.000006
stays_in_week_nights,1.385739e-16,1.000006
adults,-5.229205e-17,1.000006
children,8.889648e-17,1.000006
babies,-1.634127e-17,1.000006
is_repeated_guest,-4.444824e-17,1.000006


## 6. Save Processed Dataset

In [8]:
# Save the cleaned (pre-encoding) version for downstream notebooks
# (The pipeline will handle encoding/scaling at training time)
save_data(df_clean, PROCESSED_DATA_DIR / 'hotel_bookings_processed.csv')
print(f'✅ Processed data saved to {PROCESSED_DATA_DIR}')

2026-08-25 00:49:59 │ INFO     │ src.data.load_data │ Saved 86963 rows to C:\Users\pares\OneDrive\Desktop\hotle_cnacelation\data\processed\hotel_bookings_processed.csv
✅ Processed data saved to C:\Users\pares\OneDrive\Desktop\hotle_cnacelation\data\processed


## Summary

| Step | Before | After |
|---|---|---|
| Raw load | 119,390 × 32 | — |
| Drop columns | — | -3 columns |
| Impute nulls | 4 children, 488 country, 16K agent | 0 nulls |
| Remove invalid rows | — | ~few hundred rows removed |
| Month → numeric | string | int (1–12) |
| **Final cleaned** | — | **~118K × 29** |